# Quantitative Evaluation for `mailcom`

In [ ]:
# install Hugging Face datasets if needed
%pip install datasets

In [ ]:
from datasets import load_dataset
import json

import mailcom
import pandas as pd
import ast
import re

## Email address detection

### Data preparation

We used Hugging Face `Josephgflowers/PII-NER` dataset for this evaluation.

Since we only focus on email address detection, we first filtered the dataset to obtain a subset of sentences that contain email addresses. We then applied the `mailcom` transformation to these sentences and compared the detected email addresses with the ground-truth annotations provided in the dataset.

In [ ]:
# load the dataset
pii_ner_ds = load_dataset("Josephgflowers/PII-NER")

In [ ]:
pii_ner_ds

In [ ]:
# create a copy of the dataset that contains only the original text and the extracted email addresses

def extract_text_emails(row: str):
    try:
        assistant_output = json.loads(row["assistant"])
        text = row["user"]
        emails = assistant_output.get("EMAIL", [])
    except:
        text = row["user"]
        emails = []
    return {"text": text, "emails": emails}

email_ds = pii_ner_ds["train"].map(extract_text_emails, remove_columns=pii_ner_ds["train"].column_names)
email_ds = email_ds.filter(lambda x: (len(x["emails"]) > 0) and "@" in x["text"])
email_ds # 2216 items

In [ ]:
# email in this dataset usually followed by a comma
# for simplicity, we added spaces before and after the email addresses to separate email addresses from punctuations
def add_spaces_around_emails(text: str, emails: list):
    for email in emails:
        text = text.replace(email, f" {email} ")
    return text

email_ds = email_ds.map(lambda x: {"text": add_spaces_around_emails(x["text"], x["emails"])})

In [ ]:
# save the filtered dataset to a csv file for evaluation
email_ds.to_csv("eval/email_detection_eval.csv", index=False)

### Run `mailcom` on the evaluation dataset

In [ ]:
# load workflow configuration
new_settings = {
    "default_lang": "en", # only English in the dataset
    "pseudo_fields": ["content"], # we don't consider subject here
    "pseudo_emailaddresses": True, # we only pseudo email addresses
    "pseudo_ne": False,
    "pseudo_numbers": False,
    "datetime_detection": False,
}

# save the updated configuration to a file for reproducibility purposes
new_settings_dir = "./eval"
workflow_settings = mailcom.get_workflow_settings(new_settings=new_settings, 
                                                  updated_setting_dir= new_settings_dir,
                                                  save_updated_settings=True)

In [ ]:
# load csv file into input handler
input_csv = "eval/email_detection_eval.csv"
# the columns of the csv that should be passed through the processing pipeline/retained in the pipeline
matching_columns = ["text"]
# the predefined keys that should be used to match these columns, in the correct order
pre_defined_keys = ["content"]
# what to call any columns that are not matched to pre-defined keys
# get this from the workflow settings
unmatched_keyword = workflow_settings.get("csv_col_unmatched_keyword")

input_handler = mailcom.get_input_handler(in_path=input_csv, in_type="csv", 
                                          col_names=matching_columns, 
                                          init_data_fields=pre_defined_keys, 
                                          unmatched_keyword=unmatched_keyword)

In [ ]:
# process the input data
mailcom.process_data(input_handler.get_email_list(), workflow_settings)

In [ ]:
# convert the processed data into a dataframe
email_df = pd.DataFrame(input_handler.get_email_list())

In [ ]:
# only keep the content,  pseudo_content, and sentences columns
filtered_email_df = email_df[["content", "pseudo_content", "sentences"]]

In [ ]:
# add pseudo_content to the original dataset
org_email_df = pd.read_csv(input_csv)
# convert the emails column from string to list
org_email_df["emails"] = org_email_df["emails"].apply(ast.literal_eval)

# check before merging
(org_email_df["text"].sort_values().reset_index(drop=True) ==
 filtered_email_df["content"].sort_values().reset_index(drop=True)).all()

In [ ]:
# merge the pseudo_content into the original dataset
merged_email_df = org_email_df.merge(filtered_email_df, left_on="text", right_on="content", how="inner")

assert len(merged_email_df) == len(org_email_df) == len(filtered_email_df)

In [ ]:
merged_email_df = merged_email_df.drop(columns=["content"])
merged_email_df.head()

In [ ]:
# create expected content column as ground truth for evaluation
def clean_sentence(sentence, email_list):
    # normalize whitespace to make it match the way mailcom clean the sentences
    normalized_sentence = " ".join(re.split(r"\s+", sentence))
    return normalized_sentence
    
def replace_email_with_placeholder(sentences, email_list):
    replaced_sents = []
    for sentence in sentences:
        normalized_sentence = clean_sentence(sentence, email_list)
        for email in email_list:
            replaced_sent = normalized_sentence.replace(email, "[email]")
            replaced_sents.append(replaced_sent)
    return " ".join(replaced_sents)

merged_email_df["expected_content"] = merged_email_df.apply(lambda row: replace_email_with_placeholder(row["sentences"].get("content"), row["emails"]), axis=1)

In [ ]:
merged_email_df.head()

In [ ]:
# calculate the exact match accuracy between the pseudo_content and the expected_content
merged_email_df["exact_match"] = merged_email_df.apply(lambda row: row["pseudo_content"] == row["expected_content"], axis=1)

In [ ]:
# calculate precision, recall, and F1 score
def calculate_metrics(row):
    pred_count = len(re.findall(r"\[email\]", row["pseudo_content"]))
    gold_count = len(re.findall(r"\[email\]", row["expected_content"]))

    true_positives = min(pred_count, gold_count)
    false_positives = max(pred_count - gold_count, 0)
    false_negatives = max(gold_count - pred_count, 0)
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return pd.Series({"precision": precision, "recall": recall, "f1_score": f1_score})

metrics_df = merged_email_df.apply(calculate_metrics, axis=1)
merged_email_df = pd.concat([merged_email_df, metrics_df], axis=1)

In [ ]:
merged_email_df.head()

In [ ]:
accuracy = merged_email_df["exact_match"].mean()
print(f"Exact match accuracy: {accuracy:.4f}")
macro_precision = merged_email_df["precision"].mean()
macro_recall = merged_email_df["recall"].mean()
macro_f1 = merged_email_df["f1_score"].mean()
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")

# Exact match accuracy: 0.7243
# Macro Precision: 0.8244
# Macro Recall: 0.9224
# Macro F1 Score: 0.8570
# recall > precision, as mailcom also marks some mentioning form, e.g. @username as email addresses.

## NER detection

## Numerical data detection